# Implementing Simple Feedforward Neural Networks

This notebook explores the implementation of feedforward neural networks - the foundational architecture for many deep learning applications. We'll cover both manual implementations and how to use modern deep learning frameworks to build these networks.

## Table of Contents
1. [Import Required Libraries](#import-libraries)
2. [Understanding Feedforward Neural Networks](#understanding-feedforward)
3. [Building a Simple Neural Network from Scratch](#from-scratch)
4. [Implementing Feedforward Networks with TensorFlow/Keras](#tensorflow-keras)
5. [Implementing Feedforward Networks with PyTorch](#pytorch)
6. [Visualizing Neural Network Architecture](#visualizing)
7. [Training Process and Evaluation](#training-evaluation)
8. [Hyperparameter Tuning](#hyperparameters)
9. [Advanced Techniques for Feedforward Networks](#advanced-techniques)

## 1. Import Required Libraries <a id="import-libraries"></a>

Let's import the necessary libraries for our implementations:

In [ ]:
# NumPy for numerical operations
import numpy as np

# Matplotlib for visualization
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style('whitegrid')

# TensorFlow and Keras for neural network implementation
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print(f"TensorFlow version: {tf.__version__}")
    print(f"Keras version: {tf.keras.__version__}")
    tf_available = True
except ImportError:
    print("TensorFlow is not installed. Some sections of this notebook will not run.")
    tf_available = False

# PyTorch for neural network implementation
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    print(f"PyTorch version: {torch.__version__}")
    torch_available = True
except ImportError:
    print("PyTorch is not installed. Some sections of this notebook will not run.")
    torch_available = False

# Scikit-learn for dataset preparation and evaluation
from sklearn.datasets import make_classification, make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

# Import other utilities
import time
import random
from IPython.display import display

## 2. Understanding Feedforward Neural Networks <a id="understanding-feedforward"></a>

Feedforward Neural Networks (FNN), also known as Multi-Layer Perceptrons (MLPs), are the most basic type of neural networks. They consist of:

- **Input Layer**: Receives the raw input data
- **Hidden Layers**: Process the inputs through weighted connections
- **Output Layer**: Produces the final prediction

In a feedforward network, information flows in one direction - forward, from the input layer, through the hidden layers, to the output layer.

### Key Components

1. **Neurons/Units**: Basic computational units that apply an activation function to the weighted sum of inputs
2. **Weights and Biases**: Learnable parameters that the network optimizes during training
3. **Activation Functions**: Non-linear functions (like ReLU, sigmoid, tanh) that introduce non-linearity into the model
4. **Forward Propagation**: The process of computing outputs from inputs
5. **Backpropagation**: The algorithm that computes gradients to update weights and biases

### Mathematical Representation

For a layer with inputs $x$, weights $W$, biases $b$, and activation function $f$, the output is:

$$y = f(Wx + b)$$

Let's visualize a simple feedforward neural network structure:

In [ ]:
# Create a simple visualization of a feedforward neural network
def plot_neural_network(layer_sizes):
    # Create figure and axis
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111)
    
    # Set axis properties
    ax.axis('off')
    
    # Set the number of neurons in each layer
    n_layers = len(layer_sizes)
    
    # Maximum number of neurons in any layer (for scaling)
    max_neurons = max(layer_sizes)
    
    # Horizontal spacing between layers
    h_spacing = 1
    
    # Vertical spacing between neurons
    v_spacing = 1
    
    # Width and height of neurons
    neuron_radius = 0.2
    
    # Draw the input layer nodes
    layer_top_0 = v_spacing * (max_neurons - layer_sizes[0]) / 2. + neuron_radius
    
    # Dict to store neuron positions
    neuron_positions = {}
    
    # Draw each layer
    for l, layer_size in enumerate(layer_sizes):
        layer_top = v_spacing * (max_neurons - layer_size) / 2. + neuron_radius
        
        # Draw neurons for this layer
        for n in range(layer_size):
            x = l * h_spacing + neuron_radius
            y = layer_top + n * v_spacing
            
            # Draw the neuron
            circle = plt.Circle((x, y), radius=neuron_radius, fill=True, color='lightblue', ec='blue')
            ax.add_patch(circle)
            
            # Add layer and neuron information
            if l == 0:
                text_label = f'Input {n+1}'
            elif l == n_layers - 1:
                text_label = f'Output {n+1}'
            else:
                text_label = f'Hidden {n+1}'
                
            ax.annotate(text_label, xy=(x, y), xytext=(x, y),
                      ha='center', va='center', color='black')
            
            # Store the neuron position
            neuron_positions[(l, n)] = (x, y)
            
    # Draw the connections between layers
    for l1, layer_size_1 in enumerate(layer_sizes[:-1]):
        for l2, layer_size_2 in enumerate([layer_sizes[l1+1]]):
            layer_2_index = l1 + 1
            
            for n1 in range(layer_size_1):
                for n2 in range(layer_sizes[layer_2_index]):
                    # Draw a line between neurons
                    x1, y1 = neuron_positions[(l1, n1)]
                    x2, y2 = neuron_positions[(layer_2_index, n2)]
                    
                    line = plt.Line2D([x1, x2], [y1, y2], c='gray', alpha=0.5)
                    ax.add_line(line)
    
    # Set the limits
    plt.xlim(-0.5, n_layers * h_spacing + 0.5)
    plt.ylim(-0.5, max_neurons * v_spacing + 0.5)
    
    # Set titles
    layer_names = ["Input Layer"] + [f"Hidden Layer {i+1}" for i in range(n_layers-2)] + ["Output Layer"]
    for l, layer_name in enumerate(layer_names):
        plt.text(l * h_spacing + neuron_radius, max_neurons * v_spacing + 0.6, 
                 layer_name, ha='center', va='center', fontsize=12)
    
    plt.title('Feedforward Neural Network Architecture', fontsize=16)
    plt.tight_layout()
    plt.show()

# Example network with 4 inputs, 5 hidden neurons, 3 hidden neurons, and 2 outputs
plot_neural_network([4, 5, 3, 2])

### Activation Functions

Activation functions introduce non-linearity into the network, allowing it to learn complex patterns. Here are some common activation functions:

In [ ]:
# Define activation functions
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    return np.maximum(alpha * x, x)

# Plot activation functions
x = np.linspace(-5, 5, 100)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(x, sigmoid(x), label='Sigmoid')
plt.plot(x, tanh(x), label='Tanh')
plt.grid(True)
plt.legend()
plt.title('Sigmoid and Tanh Activation Functions')
plt.xlabel('x')
plt.ylabel('f(x)')

plt.subplot(1, 2, 2)
plt.plot(x, relu(x), label='ReLU')
plt.plot(x, leaky_relu(x), label='Leaky ReLU (alpha=0.01)')
plt.grid(True)
plt.legend()
plt.title('ReLU and Leaky ReLU Activation Functions')
plt.xlabel('x')
plt.ylabel('f(x)')

plt.tight_layout()
plt.show()

## 3. Building a Simple Neural Network from Scratch <a id="from-scratch"></a>

Let's implement a simple feedforward neural network from scratch using NumPy. This will help us understand the inner workings of neural networks before using high-level frameworks.

Our implementation will include:
1. Random initialization of weights and biases
2. Forward propagation
3. Loss computation
4. Backward propagation (gradient computation)
5. Parameter updates using gradient descent

In [ ]:
class SimpleNeuralNetwork:
    def __init__(self, layer_sizes, activation='sigmoid', learning_rate=0.01):
        """
        Initialize a simple neural network.
        
        Parameters:
        - layer_sizes: List containing the number of neurons in each layer
                      (including input and output layers)
        - activation: Activation function to use ('sigmoid', 'tanh', or 'relu')
        - learning_rate: Learning rate for gradient descent
        """
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.num_layers = len(layer_sizes)
        self.weights = []
        self.biases = []
        
        # Set activation function
        if activation == 'sigmoid':
            self.activation = sigmoid
            self.activation_derivative = lambda x: sigmoid(x) * (1 - sigmoid(x))
        elif activation == 'tanh':
            self.activation = tanh
            self.activation_derivative = lambda x: 1 - np.tanh(x)**2
        elif activation == 'relu':
            self.activation = relu
            self.activation_derivative = lambda x: np.where(x > 0, 1, 0)
        else:
            raise ValueError("Unsupported activation function")
            
        # Initialize weights and biases
        for i in range(1, self.num_layers):
            # He initialization for weights
            scale = np.sqrt(2 / layer_sizes[i-1])
            self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i-1]) * scale)
            self.biases.append(np.zeros((layer_sizes[i], 1)))
    
    def forward(self, X):
        """
        Forward propagation through the network.
        
        Parameters:
        - X: Input data of shape (n_features, n_samples)
        
        Returns:
        - activations: List of activations for each layer
        - z_values: List of weighted inputs (before activation)
        """
        # Make sure X is 2D
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        elif X.ndim > 2:
            raise ValueError("Input should be 1D or 2D")
            
        # For inputs that come in as (n_samples, n_features)
        if X.shape[0] != self.layer_sizes[0]:
            X = X.T
        
        activations = [X]  # List to store activations of each layer
        z_values = []      # List to store z values (weighted inputs)
        
        # Forward pass
        a = X
        for w, b in zip(self.weights, self.biases):
            z = np.dot(w, a) + b
            z_values.append(z)
            a = self.activation(z)
            activations.append(a)
        
        return activations, z_values
    
    def backward(self, X, y, activations, z_values):
        """
        Backward propagation to compute gradients.
        
        Parameters:
        - X: Input data
        - y: Target values
        - activations: List of activations from forward pass
        - z_values: List of weighted inputs from forward pass
        
        Returns:
        - weight_gradients: Gradients for weights
        - bias_gradients: Gradients for biases
        """
        m = X.shape[1] if X.ndim > 1 else 1
        
        # Initialize gradients
        weight_gradients = [np.zeros_like(w) for w in self.weights]
        bias_gradients = [np.zeros_like(b) for b in self.biases]
        
        # Output error (delta for output layer)
        delta = activations[-1] - y
        
        # Last layer gradients
        weight_gradients[-1] = np.dot(delta, activations[-2].T) / m
        bias_gradients[-1] = np.sum(delta, axis=1, keepdims=True) / m
        
        # Backpropagate the error
        for l in range(2, self.num_layers):
            delta = np.dot(self.weights[-l+1].T, delta) * self.activation_derivative(z_values[-l])
            weight_gradients[-l] = np.dot(delta, activations[-l-1].T) / m
            bias_gradients[-l] = np.sum(delta, axis=1, keepdims=True) / m
        
        return weight_gradients, bias_gradients
    
    def update_parameters(self, weight_gradients, bias_gradients):
        """Update weights and biases using gradient descent."""
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * weight_gradients[i]
            self.biases[i] -= self.learning_rate * bias_gradients[i]
    
    def train(self, X, y, epochs=100, batch_size=None, verbose=True):
        """
        Train the neural network.
        
        Parameters:
        - X: Input data of shape (n_samples, n_features)
        - y: Target values of shape (n_samples, n_outputs) or (n_samples,)
        - epochs: Number of training epochs
        - batch_size: Size of mini-batches (None for full batch)
        - verbose: Whether to print progress
        
        Returns:
        - history: Dictionary containing loss values during training
        """
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        
        # Reshape y if needed
        if y.ndim == 1:
            if self.layer_sizes[-1] > 1:
                # One-hot encode the targets for multi-class classification
                y_reshaped = np.zeros((len(y), self.layer_sizes[-1]))
                for i in range(len(y)):
                    y_reshaped[i, int(y[i])] = 1
                y = y_reshaped
            else:
                # Reshape for binary classification
                y = y.reshape(-1, 1)
        
        # Convert data to right format
        X = X.T
        y = y.T
        
        # For storing training history
        history = {'loss': []}
        
        # Training loop
        for epoch in range(epochs):
            if batch_size is None:
                # Full batch gradient descent
                activations, z_values = self.forward(X)
                weight_gradients, bias_gradients = self.backward(X, y, activations, z_values)
                self.update_parameters(weight_gradients, bias_gradients)
                
                # Compute loss
                loss = np.mean(np.square(activations[-1] - y))
                history['loss'].append(loss)
                
                if verbose and epoch % 100 == 0:
                    print(f"Epoch {epoch}: loss = {loss:.6f}")
            else:
                # Mini-batch gradient descent
                indices = np.random.permutation(X.shape[1])
                X_shuffled = X[:, indices]
                y_shuffled = y[:, indices]
                
                for i in range(0, X.shape[1], batch_size):
                    X_batch = X_shuffled[:, i:min(i+batch_size, X.shape[1])]
                    y_batch = y_shuffled[:, i:min(i+batch_size, y.shape[1])]
                    
                    activations, z_values = self.forward(X_batch)
                    weight_gradients, bias_gradients = self.backward(X_batch, y_batch, activations, z_values)
                    self.update_parameters(weight_gradients, bias_gradients)
                
                # Compute loss on full dataset
                activations, _ = self.forward(X)
                loss = np.mean(np.square(activations[-1] - y))
                history['loss'].append(loss)
                
                if verbose and epoch % 100 == 0:
                    print(f"Epoch {epoch}: loss = {loss:.6f}")
        
        return history
    
    def predict(self, X):
        """
        Make predictions.
        
        Parameters:
        - X: Input data of shape (n_samples, n_features)
        
        Returns:
        - predictions: Network predictions
        """
        # Ensure X is in the right format
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        
        # For multi-class classification, return class with highest probability
        activations, _ = self.forward(X.T)
        predictions = activations[-1].T
        
        if predictions.shape[1] > 1:
            # Multi-class: return index of max probability
            return np.argmax(predictions, axis=1)
        else:
            # Binary: threshold at 0.5
            return (predictions > 0.5).astype(int).flatten()

### Using our Neural Network on a Simple Problem

Let's test our implementation on the XOR problem, which is a classic example that requires non-linear separation and can't be solved by a single perceptron.

In [ ]:
# XOR data
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])  # XOR truth table

# Create and train our neural network
nn_xor = SimpleNeuralNetwork(layer_sizes=[2, 4, 1], activation='tanh', learning_rate=0.1)
history = nn_xor.train(X_xor, y_xor, epochs=10000, verbose=True)

# Plot the loss during training
plt.figure(figsize=(10, 6))
plt.plot(history['loss'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

# Make predictions
predictions = nn_xor.predict(X_xor)
print("Predictions:", predictions)
print("Actual:     ", y_xor)
print("Accuracy:", accuracy_score(y_xor, predictions))

# Visualize decision boundary
def plot_decision_boundary(X, y, model):
    # Create a meshgrid to plot the decision boundary
    h = 0.01
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Make predictions on the meshgrid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary and scatter points
    plt.figure(figsize=(8, 8))
    plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.RdBu)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap=plt.cm.RdBu)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Decision Boundary for XOR Problem')
    plt.colorbar()
    plt.grid(True)
    plt.show()

# Plot the decision boundary
plot_decision_boundary(X_xor, y_xor, nn_xor)

Let's try our model on a more complex dataset - the moons dataset from scikit-learn, which creates two interleaving half circles:

In [ ]:
# Create the moons dataset
X_moons, y_moons = make_moons(n_samples=200, noise=0.15, random_state=42)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

# Create and train our neural network
nn_moons = SimpleNeuralNetwork(layer_sizes=[2, 8, 4, 1], activation='tanh', learning_rate=0.1)
history = nn_moons.train(X_train, y_train, epochs=1000, batch_size=32, verbose=True)

# Plot the loss during training
plt.figure(figsize=(10, 6))
plt.plot(history['loss'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

# Make predictions
y_pred_train = nn_moons.predict(X_train)
y_pred_test = nn_moons.predict(X_test)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Visualize the decision boundary
plot_decision_boundary(X_moons, y_moons, nn_moons)

## 4. Implementing Feedforward Networks with TensorFlow/Keras <a id="tensorflow-keras"></a>

Now that we understand the fundamentals of feedforward neural networks, let's implement them using TensorFlow and Keras. These high-level APIs make it much easier and faster to build, train, and evaluate neural networks.

In [ ]:
if tf_available:
    # Create a simple dataset for binary classification
    X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, 
                              n_clusters_per_class=2, random_state=42)
    
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Standardize the features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Using Sequential API
    print("Building a model with Keras Sequential API")
    
    # Build the model
    model_sequential = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dropout(0.2),  # Add dropout for regularization
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')  # Output layer for binary classification
    ])
    
    # Compile the model
    model_sequential.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Display the model summary
    model_sequential.summary()
    
    # Train the model
    history_sequential = model_sequential.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )
    
    # Evaluate the model
    test_loss, test_acc = model_sequential.evaluate(X_test, y_test, verbose=0)
    print(f"Test accuracy: {test_acc:.4f}")
    
    # Using the Functional API for more flexibility
    print("\nBuilding a model with Keras Functional API")
    
    # Define inputs
    inputs = keras.Input(shape=(X_train.shape[1],))
    
    # Define the network topology
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    # Create the model
    model_functional = keras.Model(inputs=inputs, outputs=outputs)
    
    # Compile the model
    model_functional.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Display the model summary
    model_functional.summary()
    
    # Train the model
    history_functional = model_functional.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )
    
    # Plot the learning curves
    def plot_learning_curves(history):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Plot accuracy
        ax1.plot(history.history['accuracy'])
        ax1.plot(history.history['val_accuracy'])
        ax1.set_title('Model Accuracy')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend(['Train', 'Validation'], loc='best')
        ax1.grid(True)
        
        # Plot loss
        ax2.plot(history.history['loss'])
        ax2.plot(history.history['val_loss'])
        ax2.set_title('Model Loss')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend(['Train', 'Validation'], loc='best')
        ax2.grid(True)
        
        plt.tight_layout()
        plt.show()
        
    # Compare learning curves between the two models
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history_sequential.history['accuracy'], label='Sequential Train')
    plt.plot(history_sequential.history['val_accuracy'], label='Sequential Val')
    plt.plot(history_functional.history['accuracy'], label='Functional Train')
    plt.plot(history_functional.history['val_accuracy'], label='Functional Val')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(history_sequential.history['loss'], label='Sequential Train')
    plt.plot(history_sequential.history['val_loss'], label='Sequential Val')
    plt.plot(history_functional.history['loss'], label='Functional Train')
    plt.plot(history_functional.history['val_loss'], label='Functional Val')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Make predictions with the model
    y_pred_proba = model_sequential.predict(X_test)
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Sequential Model Predictions Accuracy: {accuracy:.4f}")
else:
    print("TensorFlow is not available. Skipping this section.")

### Building a More Complex Model for a Multi-Class Problem

Let's implement a feedforward neural network for a multi-class classification problem - recognizing handwritten digits using the MNIST dataset.

In [ ]:
if tf_available:
    # Load the digits dataset from sklearn (a smaller version of MNIST)
    digits = load_digits()
    X_digits, y_digits = digits.data, digits.target
    
    # Scale the data
    X_digits = X_digits / 16.0  # Maximum pixel value is 16
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits, test_size=0.2, random_state=42)
    
    # Create one-hot encoded labels for training
    y_train_onehot = keras.utils.to_categorical(y_train, 10)
    y_test_onehot = keras.utils.to_categorical(y_test, 10)
    
    # Display some sample images
    plt.figure(figsize=(10, 5))
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(X_digits[i].reshape(8, 8), cmap='gray')
        plt.title(f"Digit: {y_digits[i]}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Build the neural network
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')  # 10 output classes (0-9 digits)
    ])
    
    # Compile the model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Define callbacks for early stopping and learning rate reduction
    callbacks = [
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
    ]
    
    # Train the model
    history = model.fit(
        X_train, y_train_onehot,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=1
    )
    
    # Plot the learning curves
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'], loc='best')
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(['Train', 'Validation'], loc='best')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Evaluate the model
    test_loss, test_acc = model.evaluate(X_test, y_test_onehot)
    print(f"Test accuracy: {test_acc:.4f}")
    
    # Make predictions
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Display confusion matrix
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
    
    plt.figure(figsize=(10, 8))
    disp.plot(cmap='Blues')
    plt.title('Confusion Matrix')
    plt.grid(False)
    plt.show()
    
    # Show some misclassified examples
    misclassified_idx = np.where(y_pred != y_test)[0]
    
    if len(misclassified_idx) > 0:
        plt.figure(figsize=(12, 4))
        for i, idx in enumerate(misclassified_idx[:8]):  # Show up to 8 misclassified examples
            if i >= 8:
                break
            plt.subplot(2, 4, i + 1)
            plt.imshow(X_test[idx].reshape(8, 8), cmap='gray')
            plt.title(f"True: {y_test[idx]}, Pred: {y_pred[idx]}")
            plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("No misclassified examples found!")
else:
    print("TensorFlow is not available. Skipping this section.")

## 5. Implementing Feedforward Networks with PyTorch <a id="pytorch"></a>

PyTorch is another popular deep learning framework that provides a more dynamic and "pythonic" approach to building neural networks. Let's implement the same classification task using PyTorch:

In [ ]:
if torch_available:
    # Set the random seed for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Create a dataset
    X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, 
                              n_clusters_per_class=2, random_state=42)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Scale the features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Convert to PyTorch tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train)
    X_test_tensor = torch.FloatTensor(X_test)
    y_test_tensor = torch.FloatTensor(y_test)
    
    # Define a PyTorch Dataset
    class SimpleDataset(Dataset):
        def __init__(self, X, y):
            self.X = X
            self.y = y
        
        def __len__(self):
            return len(self.X)
        
        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]
    
    # Create datasets
    train_dataset = SimpleDataset(X_train_tensor, y_train_tensor)
    test_dataset = SimpleDataset(X_test_tensor, y_test_tensor)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)
    
    # Define the neural network
    class FeedForwardNN(nn.Module):
        def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
            super(FeedForwardNN, self).__init__()
            
            # Define layers
            self.fc1 = nn.Linear(input_dim, hidden_dim1)
            self.bn1 = nn.BatchNorm1d(hidden_dim1)
            self.dropout1 = nn.Dropout(0.2)
            
            self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
            self.bn2 = nn.BatchNorm1d(hidden_dim2)
            self.dropout2 = nn.Dropout(0.2)
            
            self.fc3 = nn.Linear(hidden_dim2, output_dim)
            
        def forward(self, x):
            # Forward pass
            x = self.fc1(x)
            x = self.bn1(x)
            x = torch.relu(x)
            x = self.dropout1(x)
            
            x = self.fc2(x)
            x = self.bn2(x)
            x = torch.relu(x)
            x = self.dropout2(x)
            
            x = self.fc3(x)
            x = torch.sigmoid(x)  # For binary classification
            
            return x
    
    # Instantiate the model
    input_dim = X_train.shape[1]
    model = FeedForwardNN(input_dim=input_dim, hidden_dim1=64, hidden_dim2=32, output_dim=1)
    
    # Define loss function and optimizer
    criterion = nn.BCELoss()  # Binary Cross Entropy Loss
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Print model architecture
    print(model)
    
    # Train the model
    num_epochs = 20
    train_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for inputs, targets in train_loader:
            # Zero the gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            targets = targets.view(-1, 1)  # Reshape to match output dimensions
            
            # Compute loss
            loss = criterion(outputs, targets)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        # Calculate average loss for this epoch
        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    
    # Plot training loss
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, num_epochs + 1), train_losses, marker='o')
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.show()
    
    # Evaluate the model
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            # Forward pass
            outputs = model(inputs)
            predicted = (outputs > 0.5).float()
            targets = targets.view(-1, 1)
            
            # Count correct predictions
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    
    accuracy = correct / total
    print(f'Test Accuracy: {accuracy:.4f}')
    
    # Make predictions on the entire test set
    model.eval()
    with torch.no_grad():
        y_pred_prob = model(X_test_tensor).squeeze().numpy()
        y_pred = (y_pred_prob > 0.5).astype(int)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy:.4f}')
    
    # Using PyTorch with the MNIST dataset
    print("\nImplementing a PyTorch model for multi-class classification:")
    
    # Load the digits dataset
    digits = load_digits()
    X_digits, y_digits = digits.data, digits.target
    
    # Scale the data
    X_digits = X_digits / 16.0
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits, test_size=0.2, random_state=42)
    
    # Convert to PyTorch tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.LongTensor(y_train)  # Use long tensor for class indices
    X_test_tensor = torch.FloatTensor(X_test)
    y_test_tensor = torch.LongTensor(y_test)
    
    # Create datasets
    train_dataset = SimpleDataset(X_train_tensor, y_train_tensor)
    test_dataset = SimpleDataset(X_test_tensor, y_test_tensor)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)
    
    # Define the neural network for multi-class classification
    class MultiClassNN(nn.Module):
        def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
            super(MultiClassNN, self).__init__()
            
            # Define layers
            self.fc1 = nn.Linear(input_dim, hidden_dim1)
            self.bn1 = nn.BatchNorm1d(hidden_dim1)
            self.dropout1 = nn.Dropout(0.3)
            
            self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
            self.bn2 = nn.BatchNorm1d(hidden_dim2)
            self.dropout2 = nn.Dropout(0.3)
            
            self.fc3 = nn.Linear(hidden_dim2, output_dim)
            
        def forward(self, x):
            # Forward pass
            x = self.fc1(x)
            x = self.bn1(x)
            x = torch.relu(x)
            x = self.dropout1(x)
            
            x = self.fc2(x)
            x = self.bn2(x)
            x = torch.relu(x)
            x = self.dropout2(x)
            
            x = self.fc3(x)
            # No activation function here - using CrossEntropyLoss which includes LogSoftmax
            
            return x
    
    # Instantiate the model
    input_dim = X_train.shape[1]
    model = MultiClassNN(input_dim=input_dim, hidden_dim1=128, hidden_dim2=64, output_dim=10)
    
    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Print model architecture
    print(model)
    
    # Train the model
    num_epochs = 30
    train_losses = []
    train_accs = []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, targets in train_loader:
            # Zero the gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            
            # Compute loss
            loss = criterion(outputs, targets)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
        
        # Calculate average loss and accuracy for this epoch
        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)
        
        epoch_acc = correct / total
        train_accs.append(epoch_acc)
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')
    
    # Plot training metrics
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(range(1, num_epochs + 1), train_losses, marker='o')
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(range(1, num_epochs + 1), train_accs, marker='o', color='green')
    plt.title('Training Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Evaluate the model
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            # Forward pass
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            
            # Count correct predictions
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
            
            # Store predictions and targets for confusion matrix
            all_preds.extend(predicted.numpy())
            all_targets.extend(targets.numpy())
    
    accuracy = correct / total
    print(f'Test Accuracy: {accuracy:.4f}')
    
    # Display confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
    
    plt.figure(figsize=(10, 8))
    disp.plot(cmap='Blues')
    plt.title('Confusion Matrix')
    plt.grid(False)
    plt.show()
else:
    print("PyTorch is not available. Skipping this section.")

## 6. Visualizing Neural Network Architecture <a id="visualizing"></a>

Visualization helps us understand the structure of our neural networks and interpret their behavior. Let's explore different ways to visualize our neural network architectures:

In [ ]:
# Function to visualize the weights of a neural network
def visualize_weights(model, layer_index, title):
    if not tf_available:
        print("TensorFlow is not available. Skipping this visualization.")
        return
        
    # Extract weights from the specified layer
    weights = model.layers[layer_index].get_weights()[0]
    
    # Create a figure to visualize weights
    plt.figure(figsize=(12, 8))
    
    # Plot weights as a heatmap
    plt.subplot(1, 2, 1)
    plt.imshow(weights, cmap='viridis')
    plt.colorbar()
    plt.title(f'{title} - Weight Matrix')
    
    # Plot the distribution of weights
    plt.subplot(1, 2, 2)
    plt.hist(weights.flatten(), bins=50, color='skyblue', edgecolor='black')
    plt.title(f'{title} - Weight Distribution')
    plt.xlabel('Weight Value')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Function to visualize the outputs of each layer in a network
def visualize_layer_outputs(model, X_sample):
    if not tf_available:
        print("TensorFlow is not available. Skipping this visualization.")
        return
        
    # Create a model that outputs activations from all layers
    layer_outputs = [layer.output for layer in model.layers if 'flatten' not in layer.name.lower()]
    activation_model = keras.Model(inputs=model.input, outputs=layer_outputs)
    
    # Get activations for a sample input
    activations = activation_model.predict(X_sample.reshape(1, -1))
    
    # Plot the activations for each layer
    plt.figure(figsize=(15, 10))
    
    for i, activation in enumerate(activations):
        layer_name = model.layers[i].name if i < len(model.layers) else f"Layer {i}"
        
        # Skip batch normalization and dropout layers
        if 'batch_normalization' in layer_name.lower() or 'dropout' in layer_name.lower():
            continue
        
        # Plot as heatmap
        plt.subplot(2, 2, i % 4 + 1)
        plt.imshow(activation.reshape(1, -1), cmap='viridis', aspect='auto')
        plt.title(f'Layer: {layer_name}')
        plt.colorbar()
        
        if (i + 1) % 4 == 0 and i > 0:
            plt.tight_layout()
            plt.show()
            
            # Create a new figure if we have more layers
            if i < len(activations) - 1:
                plt.figure(figsize=(15, 10))
    
    plt.tight_layout()
    plt.show()

# If we have TensorFlow available, generate visualizations
if tf_available:
    print("Visualizing Neural Network Architecture and Weights")
    
    try:
        # Define a small model for visualization
        vis_model = keras.Sequential([
            layers.Dense(64, activation='relu', input_shape=(20,)),
            layers.Dense(32, activation='relu'),
            layers.Dense(1, activation='sigmoid')
        ])
        
        # Compile the model
        vis_model.compile(optimizer='adam', loss='binary_crossentropy')
        
        # Generate some random data to visualize
        np.random.seed(42)
        X_sample = np.random.randn(1, 20)
        
        # Visualize layer outputs
        visualize_layer_outputs(vis_model, X_sample)
        
        # Visualize weights
        visualize_weights(vis_model, 0, "First Hidden Layer")
        visualize_weights(vis_model, 1, "Second Hidden Layer")
        
        # Use TensorFlow's visualization tools if available
        try:
            from tensorflow.keras.utils import plot_model
            plot_model(vis_model, to_file='model.png', show_shapes=True, show_layer_names=True)
            from IPython.display import Image
            display(Image('model.png'))
        except Exception as e:
            print(f"Could not generate model visualization with plot_model: {e}")
    
    except Exception as e:
        print(f"Error in visualization: {e}")
else:
    print("TensorFlow is not available. Skipping visualizations.")

## 7. Training Process and Evaluation

Training and evaluating feedforward neural networks involves several important steps and considerations to ensure optimal performance.

### Training Process Overview

The training of a feedforward neural network typically involves:

1. **Forward Propagation**: Input data passes through the network, layer by layer
2. **Loss Calculation**: Comparing predictions with actual values
3. **Backward Propagation**: Computing gradients of the loss with respect to parameters
4. **Parameter Updates**: Adjusting weights and biases using an optimization algorithm

### Key Training Components

#### Loss Functions

Different problems require different loss functions:

- **Binary Classification**: Binary Cross-Entropy
- **Multi-class Classification**: Categorical Cross-Entropy
- **Regression**: Mean Squared Error or Mean Absolute Error

#### Optimization Algorithms

Common optimizers include:

- **Stochastic Gradient Descent (SGD)**: Simple but may converge slowly
- **Adam**: Adaptive learning rates, generally performs well
- **RMSprop**: Good for recurrent networks
- **AdaGrad**: Adapts learning rate based on parameter history

In [ ]:
# Example of different optimizers in Keras
model.compile(optimizer='sgd', loss='binary_crossentropy')  # SGD
model.compile(optimizer='adam', loss='binary_crossentropy')  # Adam
model.compile(optimizer='rmsprop', loss='binary_crossentropy')  # RMSprop
```

### Learning Rate Scheduling

Decreasing the learning rate over time can help fine-tune the model:

- **Step decay**: Reduce learning rate by a factor after set epochs
- **Exponential decay**: Continuously decrease learning rate
- **1cycle policy**: Increase then decrease learning rate

```python
# Learning rate scheduler in Keras
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=0.0001
)

#### Batch Size Considerations

- **Larger batches**: Better gradient estimates but require more memory
- **Smaller batches**: Add noise which can help generalization
- **Batch size is an important hyperparameter**: Often 32, 64, 128, or 256

### Evaluation Techniques

#### Validation Strategies

- **Train/Validation/Test Split**: Typical 70%/15%/15% or 80%/10%/10%
- **K-Fold Cross-Validation**: Training on k-1 folds and validating on the remaining fold

In [ ]:
# K-Fold Cross-Validation implementation
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Train model
    model = create_model()
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
    
    # Evaluate
    score = model.evaluate(X_val, y_val, verbose=0)
    fold_scores.append(score)
    
print(f"Average score across folds: {np.mean(fold_scores):.4f}")

#### Performance Metrics

##### Classification Metrics:
- **Accuracy**: Overall correct predictions
- **Precision**: True Positives / (True Positives + False Positives)
- **Recall**: True Positives / (True Positives + False Negatives)
- **F1-Score**: 2 * (Precision * Recall) / (Precision + Recall)
- **ROC-AUC**: Area under Receiver Operating Characteristic curve

##### Regression Metrics:
- **Mean Squared Error (MSE)**: Average of squared differences
- **Root Mean Squared Error (RMSE)**: Square root of MSE
- **Mean Absolute Error (MAE)**: Average of absolute differences
- **R-squared**: Proportion of variance explained by the model

#### Early Stopping

Prevents overfitting by monitoring validation metrics and stopping when they start to degrade:

In [ ]:
# Early stopping implementation in Keras
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

#### Learning Curves

Plotting training and validation metrics helps diagnose overfitting or underfitting:

In [ ]:
# Plot learning curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy Curves')
plt.show()

## 8. Hyperparameter Tuning

Hyperparameter tuning is critical for optimizing neural network performance, as the right configuration can significantly improve results.

### Key Hyperparameters

#### Architecture Hyperparameters

- **Number of layers**: Deeper networks can learn more complex patterns
- **Number of neurons per layer**: More neurons capture more features but risk overfitting
- **Activation functions**: ReLU, sigmoid, tanh, etc.

#### Training Hyperparameters

- **Learning rate**: Step size for parameter updates (typically 0.1 to 0.0001)
- **Batch size**: Number of samples processed before updating weights
- **Number of epochs**: How many times the model processes the entire dataset
- **Optimizer choice**: SGD, Adam, RMSprop, etc.

#### Regularization Hyperparameters

- **Dropout rate**: Probability of deactivating neurons during training (typically 0.2-0.5)
- **L1/L2 regularization strength**: Controls weight decay to prevent overfitting

### Tuning Methods

#### Manual Tuning

Steps for effective manual tuning:
1. Start with default values from literature
2. Tune learning rate first (most impactful)
3. Adjust network architecture
4. Fine-tune regularization parameters
5. Keep track of experiments

#### Grid Search

Exhaustively searches through a specified parameter grid:

In [ ]:
# Grid search example with scikit-learn
from sklearn.model_selection import GridSearchCV
from keras.wrappers.scikit_learn import KerasClassifier

def create_model(neurons=64, activation='relu', dropout=0.2, lr=0.001):
    model = keras.Sequential([
        keras.layers.Dense(neurons, activation=activation, input_shape=(X_train.shape[1],)),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create the wrapped model
model = KerasClassifier(build_fn=create_model, epochs=50, batch_size=32, verbose=0)

# Define the grid search parameters
param_grid = {
    'neurons': [32, 64, 128],
    'activation': ['relu', 'tanh'],
    'dropout': [0.1, 0.2, 0.3],
    'lr': [0.01, 0.001, 0.0001],
    'batch_size': [16, 32, 64]
}

# Grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)
grid_result = grid.fit(X_train, y_train)

print(f"Best parameters: {grid_result.best_params_}")
print(f"Best score: {grid_result.best_score_:.4f}")

#### Random Search

More efficient than grid search for high-dimensional hyperparameter spaces:

In [ ]:
# Random search example with scikit-learn
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

param_dist = {
    'neurons': randint(32, 256),
    'activation': ['relu', 'tanh'],
    'dropout': uniform(0.1, 0.4),
    'lr': uniform(0.0001, 0.01),
    'batch_size': [16, 32, 64, 128]
}

# Random search
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    random_state=42
)
random_search.fit(X_train, y_train)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best score: {random_search.best_score_:.4f}")

#### Bayesian Optimization

Uses probabilistic model to efficiently search hyperparameter space:

In [ ]:
# Example with Optuna library
import optuna

def objective(trial):
    # Define hyperparameters to optimize
    neurons = trial.suggest_int('neurons', 32, 256)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    
    # Build and train model
    model = create_model(neurons=neurons, dropout=dropout, lr=lr)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=batch_size,
        verbose=0
    )
    
    # Return validation accuracy
    return history.history['val_accuracy'][-1]

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"Best parameters: {study.best_params}")
print(f"Best validation accuracy: {study.best_value:.4f}")

### Practical Tuning Tips

1. **Start small**: Begin with a simple model and gradually increase complexity
2. **Parameter budgeting**: Focus more resources on layers that need it most
3. **Learning rate warmup**: Start with smaller learning rate and gradually increase
4. **Ensembling**: Combine multiple models with different hyperparameters
5. **Track experiments**: Keep detailed records of all experiments to identify patterns

## 9. Advanced Techniques for Feedforward Networks

While simple feedforward networks are powerful, several advanced techniques can further enhance their performance.

### Advanced Activation Functions

Beyond basic activation functions like ReLU:

- **Leaky ReLU**: Prevents "dying ReLU" problem with small negative slope
- **ELU (Exponential Linear Unit)**: Smoother, can provide more robust learning
- **SELU (Scaled ELU)**: Self-normalizing properties
- **Swish (x * sigmoid(x))**: Often outperforms ReLU in deeper networks
- **GELU**: Used in transformers, smooth approximation of ReLU

In [ ]:
# Implementing advanced activations in Keras
from tensorflow.keras.layers import LeakyReLU, ELU
from tensorflow.keras import backend as K

# Model with LeakyReLU
model = keras.Sequential([
    keras.layers.Dense(64, input_shape=(input_dim,)),
    LeakyReLU(alpha=0.1),
    keras.layers.Dense(32),
    LeakyReLU(alpha=0.1),
    keras.layers.Dense(1, activation='sigmoid')
])

# Swish activation as custom function
def swish(x):
    return x * K.sigmoid(x)

keras.utils.get_custom_objects().update({'swish': keras.layers.Activation(swish)})

### Advanced Normalization Techniques

#### Batch Normalization
Normalizes layer inputs for each mini-batch, accelerating training:

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(64, input_shape=(input_dim,)),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

#### Layer Normalization
Normalizes inputs across features instead of batch dimension, useful when batch size is small:

In [ ]:
from tensorflow.keras.layers import LayerNormalization

model = keras.Sequential([
    keras.layers.Dense(64, input_shape=(input_dim,)),
    LayerNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
## Advanced Weight Initialization

- **He initialization**: Designed for ReLU activations, scales by sqrt(2/n_inputs)
- **Xavier/Glorot initialization**: Good for sigmoid/tanh, scales by sqrt(2/(n_inputs+n_outputs))
- **Orthogonal initialization**: Creates weight matrices with orthogonal rows

In [ ]:
# He initialization in Keras
keras.layers.Dense(64, kernel_initializer='he_normal', activation='relu')

# Glorot initialization in Keras
keras.layers.Dense(64, kernel_initializer='glorot_uniform', activation='tanh')

# Orthogonal initialization
keras.layers.Dense(64, kernel_initializer='orthogonal', activation='relu')

### Skip Connections

Inspired by ResNet, skip connections help information flow through deep networks:

In [ ]:
# Skip connection in Keras
inputs = keras.Input(shape=(input_dim,))
x = keras.layers.Dense(64, activation='relu')(inputs)
y = keras.layers.Dense(64, activation='relu')(x)
z = keras.layers.add([x, y])  # Skip connection
outputs = keras.layers.Dense(1, activation='sigmoid')(z)
model = keras.Model(inputs=inputs, outputs=outputs)

### Advanced Regularization Techniques

#### Dropout Variations

- **Standard Dropout**: Randomly sets units to zero during training
- **Spatial Dropout**: Drops entire feature maps in CNNs
- **Alpha Dropout**: Designed to maintain mean and variance with SELU activations
- **DropConnect**: Drops weights instead of activations

In [ ]:
# Alpha dropout (works well with SELU)
model.add(keras.layers.AlphaDropout(0.2))

#### Weight Constraints

- **Max Norm**: Limits the magnitude of weight vectors
- **Unit Norm**: Constrains weights to have unit norm

In [ ]:
from tensorflow.keras.constraints import MaxNorm, UnitNorm

# Max norm constraint
keras.layers.Dense(64, kernel_constraint=MaxNorm(3))

# Unit norm constraint
keras.layers.Dense(64, kernel_constraint=UnitNorm())

### Transfer Learning for Feedforward Networks

While more common in CNNs, transfer learning can be applied to FFNs:

1. Train a network on a large related dataset
2. Remove the output layer
3. Add new output layer(s) for your specific task
4. Freeze or fine-tune the pre-trained layers

In [ ]:
# Transfer learning example
# 1. Load pre-trained model
base_model = keras.models.load_model('pretrained_model.h5')

# 2. Remove the output layer
pretrained_layers = base_model.layers[:-1]

# 3. Create new model with same inputs but different outputs
inputs = keras.Input(shape=base_model.input_shape[1:])
x = inputs
for layer in pretrained_layers:
    layer.trainable = False  # Freeze pre-trained layers
    x = layer(x)

# Add new task-specific layers
x = keras.layers.Dense(32, activation='relu')(x)
outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

# Create new model
transfer_model = keras.Model(inputs=inputs, outputs=outputs)

### Model Distillation

Transfer knowledge from a larger "teacher" model to a smaller "student" model:

In [ ]:
# Knowledge distillation
# Teacher model (already trained)
teacher_model = create_large_model()

# Student model (smaller architecture)
student_model = create_small_model()

# Generate soft targets from teacher
soft_targets = teacher_model.predict(X_train)

# Temperature parameter for softening probability distributions
temperature = 3

# Train student using both true labels and soft targets
def distillation_loss(y_true, y_pred):
    # Hard loss on true labels
    hard_loss = keras.losses.categorical_crossentropy(y_true, y_pred)
    
    # Soft loss on teacher predictions
    soft_loss = keras.losses.categorical_crossentropy(
        keras.activations.softmax(soft_targets / temperature),
        keras.activations.softmax(y_pred / temperature)
    )
    
    return hard_loss * 0.7 + soft_loss * 0.3

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

## Conclusion: Future of Feedforward Networks

While more complex architectures like CNNs, RNNs, and Transformers have gained prominence, feedforward neural networks remain fundamental in deep learning:

### Key Takeaways

1. **Foundation of Deep Learning**: Feedforward networks provide the basic architecture upon which more complex models are built.

2. **Versatility**: Simple feedforward networks can solve a wide range of problems with proper tuning and design.

3. **Efficiency**: For many tabular data problems, well-tuned feedforward networks often outperform more complex architectures.

4. **Interpretability**: Simpler architectures are generally easier to interpret and debug than very deep or specialized networks.

### Best Practices Summary

1. **Start Simple**: Begin with a basic architecture and gradually increase complexity as needed.

2. **Properly Preprocess Data**: Normalize features and handle missing values appropriately.

3. **Thoughtful Architecture Design**: Choose appropriate layer sizes, activation functions, and initializations.

4. **Methodical Hyperparameter Tuning**: Systematic exploration of learning rates, regularization parameters, etc.

5. **Proper Evaluation**: Use cross-validation and appropriate metrics for your problem.

6. **Regularization**: Apply dropout, weight decay, and early stopping to prevent overfitting.

7. **Ensembling**: Combine multiple models for better performance and robustness.

### Future Directions

1. **Integration with Attention Mechanisms**: Combining feedforward networks with attention for better feature selection.

2. **Neural Architecture Search**: Automated discovery of optimal feedforward architectures.

3. **Hybrid Models**: Integration with symbolic AI and domain knowledge.

4. **Low-Resource Settings**: Adapting feedforward networks to work effectively with limited data and computational resources.

Feedforward neural networks will continue to be valuable as both standalone models and components of more complex architectures, especially as techniques for training, regularization, and architecture design continue to advance.